In [19]:
import torch
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt

In [20]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [21]:
class DenseNetwork(nn.Module):
    def __init__(self, layers, num_inputs=1, num_outputs=4):
        super().__init__()

        modules = []
        in_dim = num_inputs
        for h in layers:
            modules.append(nn.Linear(in_dim, h))
            modules.append(nn.Tanh())
            in_dim = h

        modules.append(nn.Linear(in_dim, num_outputs))
        self.net = nn.Sequential(*modules)

        # remove bias and set kaimimng normal with tanh nonlinearity
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="tanh")
                nn.init.zeros_(m.bias)

    def forward(self, xy):
        return self.net(xy)

In [ ]:
LAYERS = (64, 64, 64, 64, 64, 64)

p_exit, gamma = 0.4, 1.4

x_throat = 0.5  # x_length = 1.0
R_in, R_throat, R_out = map(lambda x: np.sqrt(x / np.pi), (9, 1, 4))
wall_delta = 0.05 # This is how close any points needs to be to be on the wall
external_delta = 1e-5 # This is how close any points needs to be to be on inlet/exit/throat

In [ ]:
def get_radius_at_a_distance(x: torch.Tensor) -> torch.Tensor:
    t_convergent = x / x_throat
    t_divergent = (x - x_throat) / (1.0 - x_throat)

    r_conv = R_in + (R_throat - R_in) * (3 * t_convergent**2 - 2 * t_convergent**3)
    r_div = R_throat + (R_out - R_throat) * (3 * t_divergent**2 - 2 * t_divergent**3)

    return torch.where(x < x_throat, r_conv, r_div)


def get_area(x: torch.Tensor) -> torch.Tensor:
    return torch.pi * get_radius_at_a_distance(x) ** 2


def generate_nozzle_points(n=1000, boundary_qty=0.03):
    interior_nums, boundary_nums = (
        int(n * (1 - boundary_qty)),
        int(n * boundary_qty) // 3,
    )

    interior_points = torch.rand(interior_nums, 1, device=device, dtype=torch.float32)
    inlet_points = torch.zeros(boundary_nums, 1, device=device, dtype=torch.float32)
    outlet_points = torch.ones(boundary_nums, 1, device=device, dtype=torch.float32)
    throat_points = torch.full(
        (boundary_nums, 1), x_throat, device=device, dtype=torch.float32
    )

    all_points = torch.cat(
        [interior_points, inlet_points, outlet_points, throat_points], dim=0
    )

    return all_points.requires_grad_(True)

In [ ]:
# def _mk_tensor(type, *args, **kwargs) -> torch.Tensor:
#     if type == "zeros":
#         return torch.zeros(
#             *args, dtype=torch.float32, device=device, requires_grad=True, **kwargs
#         )
#     elif type == "ones":
#         return torch.ones(
#             *args, dtype=torch.float32, device=device, requires_grad=True, **kwargs
#         )
#     elif type == "linspace":
#         return torch.linspace(
#             *args, dtype=torch.float32, device=device, requires_grad=True, **kwargs
#         )
#     elif type == "rand":
#         return torch.rand(
#             *args, dtype=torch.float32, device=device, requires_grad=True, **kwargs
#         )
#     return torch.tensor(
#         *args, dtype=torch.float32, device=device, requires_grad=True, **kwargs
#     )
# 
# def generate_nozzle_points(
#     N_interior=1000,
#     N_inlet=100,
#     N_outlet=100,
#     N_wall=200,
#     N_center=100,
# ):
#     inlet_pts = torch.hstack(
#         [
#             _mk_tensor("zeros", (N_inlet, 1)),
#             _mk_tensor("linspace", -R_in, R_in, N_inlet)[:, None],
#         ]
#     )

#     outlet_pts = torch.hstack(
#         [
#             _mk_tensor("ones", (N_outlet, 1)),
#             _mk_tensor("linspace", -R_out, R_out, N_outlet)[:, None],
#         ]
#     )

#     center_pts = torch.hstack(
#         [
#             _mk_tensor("linspace", 0, 1, N_center)[:, None],
#             _mk_tensor("zeros", (N_center, 1)),
#         ]
#     )

#     wall_x = _mk_tensor("linspace", 0, 1, N_wall)
#     wall_pts = torch.hstack(
#         [
#             wall_x[:, None],
#             get_radius_at_a_distance(wall_x)[:, None],
#         ]
#     )

#     random_interior_points = _mk_tensor("rand", N_interior, 2)
#     wall_limits = get_radius_at_a_distance(random_interior_points[:, 0])
#     interior_pts = torch.hstack(
#         [
#             random_interior_points[:, 0][:, None],
#             (random_interior_points[:, 1] * wall_limits)[:, None],
#         ]
#     )

#     points = torch.cat(
#         [interior_pts, inlet_pts, outlet_pts, wall_pts, center_pts],
#         dim=0,
#     )
#     points.requires_grad_(True)
#     return points

In [25]:
def d_(f, x):
    return torch.autograd.grad(
        f, x, grad_outputs=torch.ones_like(f), create_graph=True
    )[0]


def model_grads(model: DenseNetwork, x: torch.Tensor):
    outputs = model(x)
    rho, u, p, t = outputs[:, 0:1], outputs[:, 1:2], outputs[:, 2:3], outputs[:, 3:4]
    A = get_area(x)

    rho_x = d_(rho, x)
    u_x = d_(u, x)
    p_x = d_(p, x)
    t_x = d_(t, x)
    A_x = d_(A, x)

    return {
        "x": x,
        "rho": rho,
        "u": u,
        "p": p,
        "t": t,
        "A": A,
        "rho_x": rho_x,
        "u_x": u_x,
        "p_x": p_x,
        "t_x": t_x,
        "A_x": A_x,
    }

In [ ]:
def pde_residuals(gradients: dict[str, torch.Tensor]):
    rho, u, p, t, A = (
        gradients["rho"],
        gradients["u"],
        gradients["p"],
        gradients["t"],
        gradients["A"],
    )

    rho_x, u_x, p_x, t_x, A_x = (
        gradients["rho_x"],
        gradients["u_x"],
        gradients["p_x"],
        gradients["t_x"],
        gradients["A_x"],
    )

    mass = rho * A * u_x + rho * A_x * u + rho_x * A * u
    momentum = A * (gamma * rho * u * u_x + p_x)
    energy = rho * u * A * t_x + (gamma - 1) * p * (A * u_x + A_x * u)
    ideal = p - rho * t

    losses = [mass, momentum, energy, ideal]
    return torch.stack(losses).pow(2).mean()


def boundary_residuals(gradients: dict[str, torch.Tensor]):
    x, rho, u, p, t = (
        gradients["x"],
        gradients["rho"],
        gradients["u"],
        gradients["p"],
        gradients["t"],
    )

    inlet_mask = x[:, 0] < external_delta
    inlet_rho_residual = rho[inlet_mask] - 1.0
    inlet_p_residual = p[inlet_mask] - 1.0
    inlet_t_residual = t[inlet_mask] - 1.0

    throat_mask = (x[:, 0] - x_throat).abs() < external_delta
    throat_u_residual = u[throat_mask] - 1.0

    outlet_mask = x[:, 0] > 1.0 - external_delta
    outlet_p_residual = p[outlet_mask] - p_exit

    # wall boundary conditions and center flow conditions in the absence of y?

    return (
        inlet_rho_residual.pow(2).mean()
        + inlet_p_residual.pow(2).mean()
        + inlet_t_residual.pow(2).mean()
        + throat_u_residual.pow(2).mean()
        + outlet_p_residual.pow(2).mean()
    )


def total_loss(model: DenseNetwork, points: torch.Tensor):
    gradients = model_grads(model, points)
    pde_loss = pde_residuals(gradients)
    boundary_loss = boundary_residuals(gradients)

    return pde_loss + 10.0 * boundary_loss, pde_loss, boundary_loss

In [27]:
from typing import Optional
from pathlib import Path
from datetime import datetime
from copy import deepcopy
import os


class ModelSave:
    SAVE_DIR_NAME = "saved_models_pinn"

    def __init__(self):
        file = os.path.dirname(os.path.abspath("__file__"))
        self.save_dir = Path(file).joinpath(self.SAVE_DIR_NAME)
        self.save_dir.mkdir(exist_ok=True)

    def generate_save_path(self, suffix: Optional[str] = None) -> str:
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        model_name = f"hybrid-qpinn-v2-{timestamp}"
        if suffix:
            model_name += f"-{suffix}"
        return str(self.save_dir.joinpath(model_name + ".pt"))

    @staticmethod
    def snapshot(model: nn.Module):
        return deepcopy(model.state_dict())

    @staticmethod
    def restore(model: nn.Module, snapshot: dict):
        model.load_state_dict(snapshot)

    @staticmethod
    def update_best(
        model: nn.Module,
        current_loss: float,
        best_loss: Optional[float],
        best_snapshot: Optional[dict],
    ):
        if best_loss is None or current_loss < best_loss:
            return current_loss, ModelSave.snapshot(model)
        return best_loss, best_snapshot

    @staticmethod
    def save(model: nn.Module, path: str):
        model_state = ModelSave.snapshot(model)

        torch.save(model_state, path)
        print(f"Model saved to {path}")

    @staticmethod
    def load(path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"No model found at {path}")

        model = DenseNetwork(layers=LAYERS).to(device)
        model_state = torch.load(path, map_location=device)
        ModelSave.restore(model, model_state)
        print(f"Model loaded from {path}")
        return model

In [ ]:
from typing import TypedDict


class AdamParams(TypedDict):
    epochs: int
    print_freq: int
    resample_every: int
    lr: float


def use_adam_optimizer(
    model: DenseNetwork,
    params: AdamParams,
    history: list[tuple[float, float, float]],
    best_loss: Optional[float],
    best_state: Optional[dict],
):
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    print("starting training [Adam] optimizer...")
    for epoch in range(params["epochs"]):
        if epoch % params["resample_every"] == 0:
            points = generate_nozzle_points()

        optimizer.zero_grad()
        loss, pde_loss, boundary_loss = total_loss(model, points) # type: ignore

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        best_loss, best_state = ModelSave.update_best(
            model, loss.item(), best_loss, best_state
        )

        history.append((pde_loss.item(), boundary_loss.item(), loss.item()))
        if (epoch + 1) % params["print_freq"] == 0:
            print(
                f"Epoch {epoch+1:<{len(str(params['epochs']))}}/{params['epochs']}, PDE Loss: {pde_loss.item():.6f}, Boundary Loss: {boundary_loss.item():.6f}, Total Loss: {loss.item():.6f}"
            )

    return best_loss, best_state, history

In [ ]:
class LBFGSParams(TypedDict):
    epochs: int
    print_freq: int


def use_lbfgs_optimizer(
    model: DenseNetwork,
    params: LBFGSParams,
    history: list[tuple[float, float, float]],
    best_loss: Optional[float],
    best_state: Optional[dict],
):
    optimizer = torch.optim.LBFGS(model.parameters(), line_search_fn="strong_wolfe")
    points = generate_nozzle_points()
    last_iter = {}

    def closure():
        optimizer.zero_grad()
        loss, pde_loss, boundary_loss = total_loss(model, points)
        loss.backward()

        last_iter["total"], last_iter["pde_loss"], last_iter["boundary_loss"] = (
            loss.item(),
            pde_loss.item(),
            boundary_loss.item(),
        )
        return loss

    print("starting training [LBFGS] optimizer...")
    for epoch in range(params["epochs"]):
        loss = optimizer.step(closure)
        best_loss, best_state = ModelSave.update_best(
            model, loss.item(), best_loss, best_state
        )

        history.append(
            (last_iter["pde_loss"], last_iter["boundary_loss"], last_iter["total"])
        )
        if (epoch + 1) % params["print_freq"] == 0:
            print(
                f"Epoch {epoch+1:<{len(str(params['epochs']))}}/{params['epochs']}, PDE Loss: {last_iter['pde_loss']:.6f}, Boundary Loss: {last_iter['boundary_loss']:.6f}, Total Loss: {last_iter['total']:.6f}"
            )

    return best_loss, best_state, history

In [30]:
def train(
    model: DenseNetwork,
    adam_params: AdamParams,
    lbfgs_params: LBFGSParams,
):
    best_loss, best_state = float("inf"), None
    history = []

    adam_result = use_adam_optimizer(model, adam_params, history, best_loss, best_state)
    if adam_result is None:
        raise ValueError("use_adam_optimizer returned None")

    best_loss, best_state, history = adam_result
    if best_state is not None:
        ModelSave.restore(model, best_state)
    print("[Adam] optimizer ran.", end=" ")
    ModelSave.save(model, ModelSave().generate_save_path("adam"))
    print(f"[Adam] model saved (loss: {best_loss:.6f})")

    lbfgs_result = use_lbfgs_optimizer(
        model, lbfgs_params, history, best_loss, best_state
    )
    if lbfgs_result is None:
        raise ValueError("use_lbfgs_optimizer returned None")

    best_loss, best_state, history = lbfgs_result
    if best_state is not None:
        ModelSave.restore(model, best_state)
    print("[LBFGS] optimizer ran.", end=" ")
    ModelSave.save(model, ModelSave().generate_save_path("lbfgs"))
    print(f"[LBFGS] model saved (loss: {best_loss:.6f})")

    return history

In [2]:
model = DenseNetwork(layers=LAYERS).to(device)

{
    "total params": sum(p.numel() for p in model.parameters()),
    "trainable params": sum(p.numel() for p in model.parameters() if p.requires_grad),
}

NameError: name 'DenseNetwork' is not defined

In [ ]:
loss_history = train(
    model,
    {"epochs": 2000, "print_freq": 500, "resample_every": 100, "lr": 1e-3},
    {"epochs": 200, "print_freq": 40},
)

starting training [Adam] optimizer...
Epoch 500 /2000, PDE Loss: 3.801306, Boundary Loss: 0.000000, Total Loss: 3.801306
Epoch 1000/2000, PDE Loss: 0.048615, Boundary Loss: 0.000000, Total Loss: 0.048615
Epoch 1500/2000, PDE Loss: 0.135865, Boundary Loss: 0.000000, Total Loss: 0.135865
Epoch 2000/2000, PDE Loss: 0.067671, Boundary Loss: 0.000000, Total Loss: 0.067671
[Adam] optimizer ran. Model saved to /kaggle/working/saved_models_pinn/hybrid-qpinn-v2-2026-06-12_09-25-47-adam.pt
[Adam] model saved (loss: 0.000257)
starting training [LBFGS] optimizer...
Epoch 50/50, PDE Loss: 0.000003, Boundary Loss: 0.000000, Total Loss: 0.000003
[LBFGS] optimizer ran. Model saved to /kaggle/working/saved_models_pinn/hybrid-qpinn-v2-2026-06-12_09-25-49-lbfgs.pt
[LBFGS] model saved (loss: 0.000003)


In [ ]:
from scipy.interpolate import interp1d
from typing import Literal
import pandas as pd

model.eval()

eval_points = torch.linspace(0, 1, 1000, device=device)[:, None].requires_grad_(True)
rho, u, p, t = model(eval_points).T.detach().cpu().numpy()
x = eval_points[:, 0].detach().cpu().numpy()

M = np.sqrt(np.maximum((2 / (gamma - 1)) * (1 / t - 1), 0))

reference = pd.read_csv("./area-mach-number-reference.csv")
subsonic = reference[reference["M"] <= 1.0].sort_values("A")
supersonic = reference[reference["M"] >= 1.0].sort_values("A")


def get_branch_data(to: str, branch: pd.DataFrame):
    return branch[to] if to not in ["p", "rho", "T"] else 1 / branch[to]


def get_reference(x: torch.Tensor, to: str, branch_name: str):
    area_ratio = get_area(x).detach().numpy()
    branch = supersonic if branch_name == "supersonic" else subsonic
    return interp1d(
        branch["A"],
        get_branch_data(to, branch),
        bounds_error=False,
        fill_value="extrapolate",  # type: ignore
    )(area_ratio).squeeze()

In [ ]:
from itertools import product

plt.figure(figsize=(12, 8))
plt.subplot(4, 1, 1)
r = np.sqrt(get_area(torch.tensor(x, device=device)).cpu().numpy() / np.pi)
plt.plot(x, r, label="Nozzle Radius")
plt.plot(x, -r, label="Nozzle Radius (mirrored)")
plt.title("Nozzle Shape")
plt.xlabel("x")
plt.ylabel("Radius")
plt.legend()

M_ref_sup, M_ref_sub, p_ref_sup, p_ref_sub, t_ref_sup, t_ref_sub = map(
    lambda x: get_reference(eval_points.cpu(), x[0], x[1]),
    product(["M", "p", "T"], ["subsonic", "supersonic"]),
)


plt.subplot(4, 1, 2)
plt.plot(x, M, label="Mach Number")
plt.axhline(1, color="red", linestyle="--", label="Mach 1")
plt.plot(x, M_ref_sub, label="Reference (subsonic everywhere)", linestyle="dashed")
plt.plot(x, M_ref_sup, label="Reference (supersonic everywhere)", linestyle="dashed")
plt.title("Mach Number vs x")
plt.xlabel("x")
plt.ylabel("Mach Number")
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(x, p, label="Pressure")
plt.plot(x, p_ref_sub, label="Reference (subsonic everywhere)", linestyle="dashed")
plt.plot(x, p_ref_sup, label="Reference (supersonic everywhere)", linestyle="dashed")
plt.title("Pressure vs x")
plt.xlabel("x")
plt.ylabel("Pressure")
plt.ylim(p.min() - 0.05, p.max() + 0.05)
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(x, t, label="Temperature")
plt.plot(x, t_ref_sub, label="Reference (subsonic everywhere)", linestyle="dashed")
plt.plot(x, t_ref_sup, label="Reference (supersonic everywhere)", linestyle="dashed")
plt.title("Temperature vs x")
plt.xlabel("x")
plt.ylabel("Temperature")
plt.ylim(t.min() - 0.05, t.max() + 0.05)
plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'model' is not defined

In [34]:
u[::100]

array([-1.71591295e-04,  1.09504210e-04,  4.59656585e-05,  2.52436148e-04,
       -1.52637018e-04,  2.49932753e-04, -1.69768464e-05,  1.63029181e-04,
       -4.61269869e-04,  2.40634428e-04, -1.04203355e-05, -1.74213899e-04,
       -2.05327524e-04,  1.98911177e-04,  1.66247832e-04,  4.33430541e-05,
       -3.74808442e-05,  5.91133721e-06,  1.11888396e-04,  1.63506018e-04],
      dtype=float32)